# M2 Class-Averaged BBO — Paraphrase Classes

Reads `task4_m2_paraphrase.csv` (first 20 paraphrase classes) and produces:
- Overall ASR: M2 vs Shafran baseline
- Per-class ASR breakdown
- Loss distribution at convergence
- Per-query ASR (how well M2 generalises within a class)
- Statistical significance test

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from scipy import stats

CSV   = Path('/home/ishana/scratch/results/task4_m2_paraphrase.csv')
PLOTS = Path('/home/ishana/scratch/results')

df = pd.read_csv(CSV)
df['jam_success'] = df['jam_success'].astype(int)
print(f'Rows: {len(df)}')
print(f'Variants: {df["variant"].unique()}')
print(f'Classes:  {df["class_id"].nunique()}')
df.head()

## 1. Overall ASR

In [ ]:
overall = (
    df.groupby('variant')['jam_success']
    .agg(['sum', 'count', 'mean'])
    .rename(columns={'sum': 'n_jammed', 'count': 'n_total', 'mean': 'asr'})
    .reset_index()
)
overall['asr_pct'] = (overall['asr'] * 100).round(1)
print(overall.to_string(index=False))
print()

m2_asr      = float(overall[overall['variant'] == 'm2']['asr_pct'])
shafran_asr = float(overall[overall['variant'] == 'shafran']['asr_pct'])
print(f'M2 ASR:      {m2_asr:.1f}%')
print(f'Shafran ASR: {shafran_asr:.1f}%')
print(f'M2 gain:     {m2_asr - shafran_asr:+.1f} pp')

## 2. ASR Bar Chart

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
variants = ['shafran', 'm2']
colors   = ['steelblue', 'coral']
labels   = ['Shafran baseline\n(single-query BBO)', 'M2 class-averaged\nBBO']

asrs = [
    float(overall[overall['variant'] == v]['asr_pct'])
    for v in variants
]
bars = ax.bar(labels, asrs, color=colors, width=0.5, edgecolor='white')
for bar, asr in zip(bars, asrs):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            f'{asr:.1f}%', ha='center', va='bottom', fontsize=12, fontweight='bold')
ax.set_ylim(0, 105)
ax.set_ylabel('Attack Success Rate (%)')
ax.set_title('M2 vs Shafran Baseline — Paraphrase Classes (n=20)', fontsize=12)
plt.tight_layout()
plt.savefig(PLOTS / 'm2_overall_asr.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Per-class ASR

In [ ]:
per_class = (
    df.groupby(['class_id', 'variant'])['jam_success']
    .mean()
    .unstack('variant')
    .reset_index()
)
per_class['delta'] = per_class['m2'] - per_class['shafran']
per_class = per_class.sort_values('delta', ascending=False)
per_class['m2_pct']      = (per_class['m2']      * 100).round(1)
per_class['shafran_pct'] = (per_class['shafran']  * 100).round(1)

print('M2 > Shafran:', (per_class['delta'] > 0).sum())
print('M2 = Shafran:', (per_class['delta'] == 0).sum())
print('M2 < Shafran:', (per_class['delta'] < 0).sum())
print()
print(per_class[['class_id', 'shafran_pct', 'm2_pct', 'delta']].to_string(index=False))

In [ ]:
x = np.arange(len(per_class))
w = 0.35

fig, ax = plt.subplots(figsize=(14, 5))
ax.bar(x - w/2, per_class['shafran_pct'], w, label='Shafran', color='steelblue', alpha=0.85)
ax.bar(x + w/2, per_class['m2_pct'],      w, label='M2',      color='coral',     alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(per_class['class_id'], rotation=45, ha='right', fontsize=8)
ax.set_ylabel('ASR within class (%)')
ax.set_title('Per-class ASR: M2 vs Shafran (sorted by M2 advantage)')
ax.legend()
plt.tight_layout()
plt.savefig(PLOTS / 'm2_per_class_asr.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Final Loss Distribution

In [ ]:
# One final_loss per (class, variant) — deduplicate by taking first query_idx==0
losses = df[df['query_idx'] == 0][['variant', 'class_id', 'final_loss', 'n_iterations']]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, metric, title in [
    (axes[0], 'final_loss', 'Final BBO Loss'),
    (axes[1], 'n_iterations', 'Iterations to convergence'),
]:
    for variant, color in [('shafran', 'steelblue'), ('m2', 'coral')]:
        sub = losses[losses['variant'] == variant][metric]
        ax.hist(sub, bins=15, alpha=0.6, color=color, label=variant)
    ax.set_title(title)
    ax.set_xlabel(metric)
    ax.set_ylabel('Count')
    ax.legend()

plt.tight_layout()
plt.savefig(PLOTS / 'm2_loss_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print('Mean final loss:')
print(losses.groupby('variant')['final_loss'].describe().round(4))

## 5. Generalisation: Original vs Paraphrase Queries

In [ ]:
# query_idx==0 is the original query; idx 1-5 are paraphrases
df['query_type'] = df['query_idx'].apply(lambda i: 'original' if i == 0 else 'paraphrase')

gen = (
    df.groupby(['variant', 'query_type'])['jam_success']
    .agg(['sum', 'count', 'mean'])
    .rename(columns={'sum': 'n', 'count': 'total', 'mean': 'asr'})
    .reset_index()
)
gen['asr_pct'] = (gen['asr'] * 100).round(1)
print(gen.to_string(index=False))

fig, ax = plt.subplots(figsize=(7, 4))
x = np.arange(2)  # original / paraphrase
w = 0.3
for i, (variant, color) in enumerate([('shafran', 'steelblue'), ('m2', 'coral')]):
    sub = gen[gen['variant'] == variant].set_index('query_type')
    vals = [sub.loc[qt, 'asr_pct'] for qt in ['original', 'paraphrase']]
    ax.bar(x + i*w - w/2, vals, w, label=variant, color=color, alpha=0.85)

ax.set_xticks(x)
ax.set_xticklabels(['Original query', 'Paraphrase queries'])
ax.set_ylabel('ASR (%)')
ax.set_title('Generalisation: M2 vs Shafran on original vs paraphrase queries')
ax.legend()
plt.tight_layout()
plt.savefig(PLOTS / 'm2_generalisation.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Statistical Test (McNemar)

In [ ]:
from statsmodels.stats.contingency_tables import mcnemar

# Align M2 and Shafran per (class_id, query_idx)
m2_df      = df[df['variant'] == 'm2'].set_index(['class_id', 'query_idx'])['jam_success']
shafran_df = df[df['variant'] == 'shafran'].set_index(['class_id', 'query_idx'])['jam_success']

common_idx = m2_df.index.intersection(shafran_df.index)
m2_v      = m2_df.loc[common_idx].values.astype(int)
shafran_v = shafran_df.loc[common_idx].values.astype(int)

# Contingency table: [both_fail, shafran_yes_m2_no, shafran_no_m2_yes, both_yes]
both_fail    = int(((m2_v == 0) & (shafran_v == 0)).sum())
only_shafran = int(((m2_v == 0) & (shafran_v == 1)).sum())
only_m2      = int(((m2_v == 1) & (shafran_v == 0)).sum())
both_success = int(((m2_v == 1) & (shafran_v == 1)).sum())

table = [[both_fail, only_shafran], [only_m2, both_success]]
print('Contingency table:')
print(f'              Shafran fail  Shafran success')
print(f'M2 fail       {both_fail:>12}  {only_shafran:>15}')
print(f'M2 success    {only_m2:>12}  {both_success:>15}')
print()

result = mcnemar(table, exact=True)
print(f'McNemar p-value: {result.pvalue:.4f}')
if result.pvalue < 0.05:
    print('M2 is significantly better than Shafran (p<0.05).')
else:
    print('No significant difference detected (p>=0.05).')

## 7. Summary

In [ ]:
print('=' * 60)
print('M2 CLASS-AVERAGED BBO — SUMMARY (20 paraphrase classes)')
print('=' * 60)
print()
print(f'{'Variant':<14} {'ASR':>8}  Notes')
print('-' * 50)
for _, row in overall.iterrows():
    print(f"{row['variant']:<14} {row['asr_pct']:>7.1f}%")
print()
print(f'M2 improvement: {m2_asr - shafran_asr:+.1f} pp')
print()

m2_orig = float(gen[(gen['variant']=='m2') & (gen['query_type']=='original')]['asr_pct'])
m2_para = float(gen[(gen['variant']=='m2') & (gen['query_type']=='paraphrase')]['asr_pct'])
print(f'M2 ASR — original queries:   {m2_orig:.1f}%')
print(f'M2 ASR — paraphrase queries: {m2_para:.1f}%')
print(f'Generalisation gap:          {m2_para - m2_orig:+.1f} pp')
print()

m2_iters = float(losses[losses['variant']=='m2']['n_iterations'].mean())
sh_iters = float(losses[losses['variant']=='shafran']['n_iterations'].mean())
print(f'Mean iterations — M2:      {m2_iters:.0f}')
print(f'Mean iterations — Shafran: {sh_iters:.0f}')
print('=' * 60)